# 📊 Enriquecimiento de Noticias Financieras con Yahoo Finance

**Pipeline completo:** `noticias_limpio.json` → enriquecimiento bursátil → `noticias_enriquecido.json`

---
**Secciones:**
1. Instalación de dependencias
2. Configuración global
3. Lectura del JSON de entrada
4. Mapeo índices / sectores → tickers
5. Descarga de datos de Yahoo Finance
6. Funciones de cálculo de retornos
7. Enriquecimiento de noticias
8. Guardado de `noticias_enriquecido.json`
9. Estadísticas finales

## 1. 📦 Instalación de dependencias

In [ ]:
# Instalar / actualizar las librerías necesarias
# NOTA: yfinance se fija a 0.2.x para garantizar compatibilidad.
#       yfinance >= 1.0 cambió la API de columnas (Close → Price en algunos contextos).
!pip install -q "yfinance>=0.2.36,<1.0" pandas tqdm pandas_market_calendars
print("✅ Dependencias instaladas correctamente.")


## 2. ⚙️ Configuración global

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import json
import logging
import time
import traceback
from datetime import datetime, timedelta, date
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pandas as pd
import yfinance as yf
from tqdm import tqdm
import pandas_market_calendars as mcal

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURACIÓN DE LOGGING
# ─────────────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  [%(levelname)s]  %(message)s",
    handlers=[
        logging.FileHandler("enriquecimiento_errores.log", mode="w", encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger(__name__)

# ─────────────────────────────────────────────────────────────────────────────
# PARÁMETROS GLOBALES
# ─────────────────────────────────────────────────────────────────────────────
INPUT_FILE: str  = "noticias_limpio.json"      # Fichero de entrada
OUTPUT_FILE: str = "noticias_enriquecido.json" # Fichero de salida

MAX_RETRIES: int   = 3    # Intentos máximos por descarga
RETRY_DELAY: float = 2.0  # Segundos entre reintentos

# Calendario bursátil que se usará para determinar días de mercado.
# NYSE cubre S&P500, NASDAQ y los ETFs de sectores americanos.
# Para el EURO STOXX 50 usaremos el calendario de Euronext.
NYSE_CAL   = mcal.get_calendar("NYSE")
EUREX_CAL  = mcal.get_calendar("EUREX")

# Horizontes temporales (en días bursátiles)
HORIZONS: Dict[str, int] = {"t1": 1, "t5": 5, "t20": 20}

# Umbrales para clasificación de reacción de mercado (%)
BULLISH_THRESHOLD: float =  2.0
BEARISH_THRESHOLD: float = -2.0

print("✅ Configuración cargada.")
logger.info("Pipeline de enriquecimiento iniciado.")

## 2.1 📤 Carga manual del archivo JSON

Sube directamente el archivo de noticias de entrada (`noticias_limpio.json`, o el
nombre que le hayas dado tras la Fase 2 de QA) — sin necesidad de tenerlo ya en el
entorno de Colab ni de montar Google Drive. Esto sobrescribe `INPUT_FILE` y deriva
`OUTPUT_FILE` a partir del nombre subido.


In [ ]:
# --- Detección de entorno (Colab vs. local) ----------------------------------
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("No estamos en Google Colab: se omite el selector de subida.")

import os

if IN_COLAB:
    print("Sube tu archivo JSON de noticias (salida de la Fase 2 de QA):")
    subido = files.upload()
    if subido:
        INPUT_FILE = list(subido.keys())[0]
        print(f"Archivo recibido: {INPUT_FILE}")
    else:
        raise RuntimeError("No se subió ningún archivo. Vuelve a ejecutar la celda para intentarlo de nuevo.")
else:
    # Fuera de Colab: se conserva el INPUT_FILE ya definido en la configuración
    # global, siempre que exista localmente.
    if not os.path.exists(INPUT_FILE):
        raise FileNotFoundError(
            f"No se encontró '{INPUT_FILE}' localmente. Coloca tu archivo de entrada "
            f"en el directorio actual o ajusta la variable INPUT_FILE manualmente."
        )
    print(f"Usando archivo local: {INPUT_FILE}")

# OUTPUT_FILE se deriva del nombre del archivo de entrada, para que quede
# claro a qué lote de noticias enriquecidas corresponde.
_nombre_base, _ext = os.path.splitext(INPUT_FILE)
if not _ext:
    _ext = ".json"
OUTPUT_FILE = f"{_nombre_base}_enriquecido{_ext}"

print(f"INPUT_FILE  = {INPUT_FILE}")
print(f"OUTPUT_FILE = {OUTPUT_FILE}")


## 3. 📂 Lectura del JSON de entrada

In [ ]:
def load_news(filepath: str) -> List[Dict]:
    """
    Carga el fichero JSON de noticias.

    Parameters
    ----------
    filepath : str
        Ruta al fichero JSON de entrada.

    Returns
    -------
    List[Dict]
        Lista de diccionarios, uno por noticia.

    Raises
    ------
    FileNotFoundError
        Si el fichero no existe en la ruta indicada.
    """
    path = Path(filepath)
    if not path.exists():
        raise FileNotFoundError(
            f"No se encontró el fichero '{filepath}'.\n"
            "Si estás en Google Colab, sube el fichero desde el panel izquierdo "
            "(icono de carpeta → 'Upload') o monta Google Drive."
        )
    with open(path, "r", encoding="utf-8") as fh:
        data = json.load(fh)
    logger.info("Fichero '%s' cargado: %d noticias.", filepath, len(data))
    return data


# ── Carga ──────────────────────────────────────────────────────────────────
# NOTA COLAB: Si el fichero no está en el directorio de trabajo (/content),
# descomenta y ajusta una de las siguientes opciones:
#
# Opción A – Subir manualmente:
#   from google.colab import files
#   uploaded = files.upload()   # Selecciona noticias_limpio.json
#
# Opción B – Desde Google Drive:
#   from google.colab import drive
#   drive.mount('/content/drive')
#   INPUT_FILE = '/content/drive/MyDrive/noticias_limpio.json'

noticias: List[Dict] = load_news(INPUT_FILE)

print(f"\n📰 Total de noticias cargadas: {len(noticias)}")
print("\n🔎 Ejemplo de registro:")
sample = {k: v for k, v in noticias[0].items() if k != 'contenido'}
print(json.dumps(sample, ensure_ascii=False, indent=2))

## 4. 🗺️ Mapeo índices / sectores → tickers

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MAPA: valor de 'indice_sector' en el JSON  →  ticker de Yahoo Finance
# ─────────────────────────────────────────────────────────────────────────────
#
# Índices bursátiles
#   'S&P 500'       → ^GSPC    (S&P 500)
#   'NASDAQ 100'    → ^NDX     (NASDAQ-100)
#   'Euro Stoxx 50' → ^STOXX50E (Euro Stoxx 50, Euronext)
#
# ETFs de sectores (todos cotizan en NYSE / NASDAQ)
#   'Sector Energía'            → XLE  (Energy Select Sector SPDR)
#   'Sector Turismo/Aerolíneas' → JETS (U.S. Global Jets ETF)
#   'Sector Defensa'            → ITA  (iShares U.S. Aerospace & Defense ETF)
#   'Sector Semiconductores'    → SMH  (VanEck Semiconductor ETF)

TICKER_MAP: Dict[str, str] = {
    # ── Índices ──────────────────────────────────────────────────────────────
    "S&P 500":       "^GSPC",
    "NASDAQ 100":    "^NDX",
    "Euro Stoxx 50": "^STOXX50E",
    # ── ETFs de sectores ─────────────────────────────────────────────────────
    "Sector Energía":            "XLE",
    "Sector Turismo/Aerolíneas": "JETS",
    "Sector Defensa":            "ITA",
    "Sector Semiconductores":    "SMH",
}

# Calendario bursátil por ticker:
# Los ETFs americanos y los índices americanos usan NYSE.
# El Euro Stoxx 50 usa el calendario de Euronext (EUREX).
CALENDAR_MAP: Dict[str, str] = {
    "^GSPC":     "NYSE",
    "^NDX":      "NYSE",
    "^STOXX50E": "EUREX",
    "XLE":       "NYSE",
    "JETS":      "NYSE",
    "ITA":       "NYSE",
    "SMH":       "NYSE",
}

# Fecha de inicio de cotización de cada instrumento.
# Las noticias anteriores a estas fechas recibirán enrich_status = 'pre_inception'
# en lugar de 'no_price', lo que permite filtrarlas correctamente en análisis posteriores.
from datetime import date as _date
ETF_INCEPTION: Dict[str, _date] = {
    "^GSPC":     _date(1993,  1, 29),   # S&P 500 (datos yfinance desde ~1927, pero 1993 es seguro)
    "^NDX":      _date(1985, 10,  1),   # NASDAQ-100
    "^STOXX50E": _date(1998,  1,  1),   # Euro Stoxx 50
    "XLE":       _date(1998, 12, 22),   # Energy Select Sector SPDR
    "JETS":      _date(2015,  4, 28),   # U.S. Global Jets ETF
    "ITA":       _date(2006,  5,  5),   # iShares U.S. Aerospace & Defense ETF
    "SMH":       _date(2000,  5,  5),   # VanEck Semiconductor ETF
}


def get_ticker(indice_sector: Optional[str]) -> Optional[str]:
    """
    Devuelve el ticker de Yahoo Finance correspondiente al valor
    de 'indice_sector' del JSON de noticias.

    Parameters
    ----------
    indice_sector : str or None
        Valor del campo 'indice_sector' de la noticia.

    Returns
    -------
    str or None
        Ticker de Yahoo Finance, o None si no se reconoce el valor.
    """
    if not indice_sector:
        return None
    ticker = TICKER_MAP.get(indice_sector.strip())
    if ticker is None:
        logger.warning("Valor desconocido en 'indice_sector': '%s'", indice_sector)
    return ticker


# ── Validación rápida del mapeo ─────────────────────────────────────────────
unique_vals = {n.get("indice_sector") for n in noticias}
print("📌 Valores únicos de 'indice_sector' en el dataset:")
for v in sorted(unique_vals, key=lambda x: x or ""):
    ticker = get_ticker(v)
    status = "✅" if ticker else "❌ SIN MAPEO"
    print(f"  {status}  {v!r}  →  {ticker}")

## 5. 📥 Descarga de datos de Yahoo Finance

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CACHÉ GLOBAL de precios descargados
# ─────────────────────────────────────────────────────────────────────────────
_price_cache: Dict[str, pd.DataFrame] = {}


def _get_market_calendar(ticker: str) -> mcal.MarketCalendar:
    cal_name = CALENDAR_MAP.get(ticker, "NYSE")
    return mcal.get_calendar(cal_name)


def get_trading_days(ticker: str, start: date, n_days: int = 30) -> pd.DatetimeIndex:
    calendar = _get_market_calendar(ticker)
    end = start + timedelta(days=n_days * 2)
    schedule = calendar.schedule(
        start_date=start.strftime("%Y-%m-%d"),
        end_date=end.strftime("%Y-%m-%d"),
    )
    # mcal.date_range devuelve un DatetimeIndex tz-aware (UTC).
    # Se elimina la zona horaria para poder compararlo con pd.Timestamp(date_obj) tz-naive.
    trading = mcal.date_range(schedule, frequency="1D").normalize()
    if trading.tz is not None:
        trading = trading.tz_localize(None)
    return trading


def next_trading_day(ticker: str, ref_date: date) -> date:
    trading_days = get_trading_days(ticker, ref_date, n_days=10)
    ref_ts = pd.Timestamp(ref_date)
    future = trading_days[trading_days >= ref_ts]
    if future.empty:
        raise ValueError(f"No se encontró día bursátil para {ticker} a partir de {ref_date}")
    return future[0].date()


def trading_day_offset(ticker: str, base_date: date, offset: int) -> date:
    trading_days = get_trading_days(ticker, base_date, n_days=offset + 30)
    base_ts = pd.Timestamp(base_date)
    idx_arr = trading_days[trading_days >= base_ts]
    if len(idx_arr) <= offset:
        raise ValueError(
            f"No hay suficientes días bursátiles tras {base_date} para offset={offset}"
        )
    return idx_arr[offset].date()


def _normalize_series(s: pd.Series) -> pd.Series:
    """
    FIX BUG 3: Garantiza que el índice de la Serie sea tz-naive y esté normalizado
    a medianoche (sin componente horaria), de modo que los lookups con
    pd.Timestamp(date_obj) funcionen correctamente en get_close_price.

    yfinance puede devolver índices tz-aware (America/New_York, UTC, etc.).
    normalize() preserva la zona horaria; hay que eliminarla explícitamente
    con tz_localize(None) para que la comparación ts in series.index sea True.
    """
    idx = pd.to_datetime(s.index)
    if idx.tz is not None:
        idx = idx.tz_convert("UTC").tz_localize(None)  # → medianoche UTC tz-naive
    s = s.copy()
    s.index = idx.normalize()
    return s


def download_prices(
    ticker: str,
    start: date,
    end: date,
    retries: int = MAX_RETRIES,
    delay: float = RETRY_DELAY,
) -> pd.Series:
    """
    Descarga precios de cierre ajustados de Yahoo Finance para el ticker
    indicado en el rango [start, end]. Implementa reintentos automáticos.

    CORRECCIONES APLICADAS:
        BUG 1 – MultiIndex en yfinance >= 0.2.x:
            Tras get_level_values(0) pueden aparecer columnas duplicadas
            ('Close', 'Close', ...) cuando yfinance devuelve MultiIndex
            de un solo ticker. Se usa .squeeze() para obtener siempre
            una Series, no un DataFrame.

        BUG 2 – Caché: loc[str:str] sobre índice tz-aware:
            Se llama a _normalize_series() antes de guardar en caché y
            antes de retornar el slice, garantizando índice tz-naive.

        BUG 3 – Índice tz-aware en la Serie devuelta:
            _normalize_series() elimina la zona horaria del índice para
            que los lookups en get_close_price no fallen silenciosamente.
    """
    global _price_cache

    # ── 1. Consultar caché ──────────────────────────────────────────────────
    # FIX: La comprobación min/max no detecta HUECOS en el índice (p.ej. datos de 2008
    # y de 2020 en caché pero no los años intermedios). Se verifica que start y end
    # estén realmente presentes en el índice antes de usar la caché.
    cache_key = ticker
    if cache_key in _price_cache:
        cached_series: pd.Series = _price_cache[cache_key]
        start_ts = pd.Timestamp(start)
        end_ts   = pd.Timestamp(end)
        # Buscar con tolerancia de ±2 días (festivos/fines de semana)
        def _in_cache(ts):
            for delta in [0, 1, -1, 2, -2]:
                if (ts + pd.Timedelta(days=delta)) in cached_series.index:
                    return True
            return False
        if _in_cache(start_ts) and _in_cache(end_ts):
            return cached_series.loc[start_ts:end_ts]

    # ── 2. Descargar con reintentos ─────────────────────────────────────────
    dl_start = start - timedelta(days=5)
    dl_end   = end   + timedelta(days=5)

    for attempt in range(1, retries + 1):
        try:
            df = yf.download(
                ticker,
                start=dl_start.strftime("%Y-%m-%d"),
                end=(dl_end + timedelta(days=1)).strftime("%Y-%m-%d"),
                auto_adjust=True,
                progress=False,
                threads=False,
            )
            if df.empty:
                raise ValueError(f"Yahoo Finance devolvió datos vacíos para {ticker}")

            # BUG 1 FIX: extraer columna de cierre de forma robusta
            # yfinance puede devolver MultiIndex o columnas planas según la versión.
            # Con auto_adjust=True la columna se llama 'Close' (no 'Adj Close').
            if isinstance(df.columns, pd.MultiIndex):
                # Buscar 'Close' en el primer nivel del MultiIndex
                close_cols = [c for c in df.columns if str(c[0]).lower() == "close"]
                if not close_cols:
                    # Fallback: aplanar y buscar 'close' en cualquier forma
                    df_flat = df.copy()
                    df_flat.columns = [str(c[0]) for c in df_flat.columns]
                    close_name = next((c for c in df_flat.columns if c.lower() == "close"), None)
                    if close_name is None:
                        raise ValueError(f"No se encontró columna 'Close' en {df.columns.tolist()}")
                    close_series = df_flat[close_name].squeeze()
                else:
                    close_series = df[close_cols[0]].squeeze()
            else:
                # Columnas planas: buscar 'Close' (case-insensitive por robustez)
                close_name = next((c for c in df.columns if str(c).lower() == "close"), None)
                if close_name is None:
                    raise ValueError(f"No se encontró columna 'Close' en {df.columns.tolist()}")
                close_series = df[close_name].squeeze()

            # BUG 2+3 FIX: normalizar índice a tz-naive
            close_series = _normalize_series(close_series)
            close_series.name = "Close"

            # Actualizar caché
            if cache_key in _price_cache:
                old = _price_cache[cache_key]
                close_series = pd.concat([old, close_series])
                close_series = close_series[~close_series.index.duplicated(keep="last")]
                close_series.sort_index(inplace=True)
            _price_cache[cache_key] = close_series

            start_ts = pd.Timestamp(start)
            end_ts   = pd.Timestamp(end)
            return close_series.loc[start_ts:end_ts]

        except Exception as exc:
            logger.warning(
                "Intento %d/%d fallido para %s [%s, %s]: %s",
                attempt, retries, ticker, start, end, exc,
            )
            if attempt < retries:
                time.sleep(delay)
            else:
                logger.error("No se pudo descargar %s tras %d intentos.", ticker, retries)

    return pd.Series(dtype=float)


def get_close_price(
    ticker: str,
    target_date: date,
    prices: Optional[pd.Series] = None,
) -> Optional[float]:
    """
    Devuelve el precio de cierre para target_date.

    BUG 3 FIX: el índice de prices ya es tz-naive (garantizado por
    _normalize_series en download_prices), por lo que pd.Timestamp(target_date)
    coincide correctamente con las entradas del índice.
    """
    ts = pd.Timestamp(target_date)  # tz-naive, coincide con el índice normalizado
    if prices is not None and not prices.empty and ts in prices.index:
        val = prices.at[ts]
        # squeeze() puede devolver scalar o Series si el índice tiene duplicados
        if isinstance(val, pd.Series):
            val = val.iloc[0]
        return float(round(val, 4)) if pd.notna(val) else None
    return None


print("✅ Funciones de descarga y calendario definidas (bugs 1-3 corregidos).")


## 6. 📐 Funciones de cálculo de retornos y clasificación

In [ ]:
def compute_return(
    price_t0: Optional[float],
    price_future: Optional[float],
) -> Optional[float]:
    """
    Calcula el retorno porcentual entre dos precios.

    Fórmula:
        return = (price_future - price_t0) / price_t0 * 100

    Parameters
    ----------
    price_t0     : float or None  Precio de cierre en t0.
    price_future : float or None  Precio de cierre en t0 + N días bursátiles.

    Returns
    -------
    float or None
        Retorno en porcentaje redondeado a 4 decimales,
        o None si alguno de los precios es nulo.
    """
    if price_t0 is None or price_future is None:
        return None
    if price_t0 == 0.0:
        return None
    return round((price_future - price_t0) / price_t0 * 100, 4)


def classify_reaction(ret: Optional[float]) -> Optional[str]:
    """
    Clasifica la reacción del mercado a partir del retorno porcentual.

    Reglas:
        ret >  2.0%  →  'bullish'
        ret < -2.0%  →  'bearish'
        otro         →  'neutral'
        None         →  None

    Parameters
    ----------
    ret : float or None  Retorno porcentual.

    Returns
    -------
    str or None
    """
    if ret is None:
        return None
    if ret > BULLISH_THRESHOLD:
        return "bullish"
    if ret < BEARISH_THRESHOLD:
        return "bearish"
    return "neutral"


print("✅ Funciones de cálculo de retornos y clasificación definidas.")

## 7. 🔧 Enriquecimiento de noticias

In [ ]:
def parse_fecha(fecha_str: str) -> Optional[date]:
    """
    Parsea el campo 'fecha' del JSON a un objeto date de Python.
    Admite los formatos ISO 8601 con y sin hora/timezone.

    Parameters
    ----------
    fecha_str : str  Cadena de fecha del JSON, p. ej. '2022-06-15T18:10:00.000Z'.

    Returns
    -------
    date or None
    """
    if not fecha_str:
        return None
    try:
        ts = pd.to_datetime(fecha_str, utc=True)
        return ts.date()
    except Exception:
        try:
            return datetime.fromisoformat(fecha_str[:10]).date()
        except Exception:
            return None


def enrich_news_item(
    noticia: Dict,
    idx: int,
) -> Dict:
    """
    Enriquece una noticia individual con datos bursátiles de Yahoo Finance.

    Campos añadidos:
        ticker          : Ticker Yahoo Finance utilizado.
        price_t0        : Precio de cierre en t0 (día de publicación o siguiente sesión).
        price_t1        : Precio de cierre en t0 + 1 día bursátil.
        price_t5        : Precio de cierre en t0 + 5 días bursátiles.
        price_t20       : Precio de cierre en t0 + 20 días bursátiles.
        return_1d       : Retorno porcentual a 1 día bursátil.
        return_5d       : Retorno porcentual a 5 días bursátiles.
        return_20d      : Retorno porcentual a 20 días bursátiles.
        market_reaction_1d  : Clasificación bullish/bearish/neutral a 1d.
        market_reaction_5d  : Clasificación bullish/bearish/neutral a 5d.
        market_reaction_20d : Clasificación bullish/bearish/neutral a 20d.
        enrich_status   : 'ok' | 'error' | 'no_ticker' | 'no_date' | 'no_price'.
        enrich_error    : Mensaje de error (solo si enrich_status != 'ok').

    Parameters
    ----------
    noticia : Dict  Diccionario con los campos originales de la noticia.
    idx     : int   Índice de la noticia (para logging).

    Returns
    -------
    Dict
        Diccionario original ampliado con los nuevos campos.
    """
    # Copia para no mutar el original
    item = dict(noticia)

    # Campos por defecto (se sobreescribirán si el enriquecimiento tiene éxito)
    item.update({
        "ticker":              None,
        "price_t0":            None,
        "price_t1":            None,
        "price_t5":            None,
        "price_t20":           None,
        "return_1d":           None,
        "return_5d":           None,
        "return_20d":          None,
        "market_reaction_1d":  None,
        "market_reaction_5d":  None,
        "market_reaction_20d": None,
        "enrich_status":       "ok",
        "enrich_error":        None,
    })

    try:
        # ── A. Determinar ticker ────────────────────────────────────────────
        ticker = get_ticker(noticia.get("indice_sector"))
        if ticker is None:
            item["enrich_status"] = "no_ticker"
            item["enrich_error"]  = f"Sin ticker para '{noticia.get('indice_sector')}'"
            return item
        item["ticker"] = ticker

        # ── B. Parsear fecha ────────────────────────────────────────────────
        pub_date = parse_fecha(noticia.get("fecha", ""))
        if pub_date is None:
            item["enrich_status"] = "no_date"
            item["enrich_error"]  = f"Fecha no parseable: '{noticia.get('fecha')}'"
            return item

        # ── B2. Comprobar fecha de inicio de cotización del instrumento ──────
        # Algunos ETFs no existían en la fecha de publicación de la noticia.
        # Devolvemos 'pre_inception' para distinguirlo de errores reales.
        inception = ETF_INCEPTION.get(ticker)
        if inception is not None and pub_date < inception:
            item["enrich_status"] = "pre_inception"
            item["enrich_error"]  = (
                f"{ticker} no cotizaba el {pub_date} "
                f"(inicio de cotización: {inception})"
            )
            return item

        # ── C. Ajustar t0 al siguiente día bursátil disponible ──────────────
        t0 = next_trading_day(ticker, pub_date)

        # ── D. Calcular fechas objetivo (t+1, t+5, t+20) ───────────────────
        t1  = trading_day_offset(ticker, t0, 1)
        t5  = trading_day_offset(ticker, t0, 5)
        t20 = trading_day_offset(ticker, t0, 20)

        # ── E. Descargar precios en rango [t0, t20] ─────────────────────────
        prices = download_prices(ticker, t0, t20)
        if prices.empty:
            item["enrich_status"] = "no_price"
            item["enrich_error"]  = f"Sin datos de precio para {ticker} [{t0}, {t20}]"
            return item

        # ── F. Extraer precios de cierre ────────────────────────────────────
        p_t0  = get_close_price(ticker, t0,  prices)
        p_t1  = get_close_price(ticker, t1,  prices)
        p_t5  = get_close_price(ticker, t5,  prices)
        p_t20 = get_close_price(ticker, t20, prices)

        # Fallback: si no hay precio exacto, tomar el más cercano dentro de ±2 días
        def nearest_price(target: date, series: pd.Series) -> Optional[float]:
            ts = pd.Timestamp(target)
            if ts in series.index:
                val = series.at[ts]
                return float(round(val, 4)) if pd.notna(val) else None
            # Buscar en ±2 días
            for delta in [1, -1, 2, -2]:
                candidate = pd.Timestamp(target + timedelta(days=delta))
                if candidate in series.index:
                    val = series.at[candidate]
                    return float(round(val, 4)) if pd.notna(val) else None
            return None

        if p_t0  is None: p_t0  = nearest_price(t0,  prices)
        if p_t1  is None: p_t1  = nearest_price(t1,  prices)
        if p_t5  is None: p_t5  = nearest_price(t5,  prices)
        if p_t20 is None: p_t20 = nearest_price(t20, prices)

        if p_t0 is None:
            item["enrich_status"] = "no_price"
            item["enrich_error"]  = f"Sin precio de cierre en t0={t0} para {ticker}"
            return item

        # ── G. Almacenar precios ────────────────────────────────────────────
        item["price_t0"]  = p_t0
        item["price_t1"]  = p_t1
        item["price_t5"]  = p_t5
        item["price_t20"] = p_t20

        # ── H. Calcular retornos ────────────────────────────────────────────
        item["return_1d"]  = compute_return(p_t0, p_t1)
        item["return_5d"]  = compute_return(p_t0, p_t5)
        item["return_20d"] = compute_return(p_t0, p_t20)

        # ── I. Clasificar reacción de mercado ───────────────────────────────
        item["market_reaction_1d"]  = classify_reaction(item["return_1d"])
        item["market_reaction_5d"]  = classify_reaction(item["return_5d"])
        item["market_reaction_20d"] = classify_reaction(item["return_20d"])

        item["enrich_status"] = "ok"

    except Exception as exc:
        item["enrich_status"] = "error"
        item["enrich_error"]  = str(exc)
        logger.error(
            "Error inesperado al enriquecer noticia idx=%d titulo='%s': %s\n%s",
            idx,
            noticia.get("titulo", "N/A")[:60],
            exc,
            traceback.format_exc(),
        )

    return item


print("✅ Función de enriquecimiento definida.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BUCLE PRINCIPAL DE ENRIQUECIMIENTO
# ─────────────────────────────────────────────────────────────────────────────
# Se procesan todas las noticias de forma secuencial.
# La barra de progreso (tqdm) muestra el avance en tiempo real.
# Los errores se registran en el log sin interrumpir el proceso.

noticias_enriquecidas: List[Dict] = []
errores: List[Dict] = []

logger.info("Iniciando enriquecimiento de %d noticias...", len(noticias))

for i, noticia in enumerate(tqdm(noticias, desc="Enriqueciendo", unit="noticia")):
    resultado = enrich_news_item(noticia, idx=i)
    noticias_enriquecidas.append(resultado)

    if resultado["enrich_status"] != "ok":
        errores.append({
            "idx":    i,
            "titulo": noticia.get("titulo", "")[:80],
            "fecha":  noticia.get("fecha", ""),
            "indice_sector": noticia.get("indice_sector", ""),
            "status": resultado["enrich_status"],
            "error":  resultado.get("enrich_error", ""),
        })

n_ok     = sum(1 for n in noticias_enriquecidas if n["enrich_status"] == "ok")
n_errors = len(errores)

logger.info("Enriquecimiento completado: %d ok, %d con error.", n_ok, n_errors)
print(f"\n✅ Enriquecimiento completado.")
print(f"   ✔ OK     : {n_ok}")
print(f"   ✖ Errores: {n_errors}")

## 8. 💾 Guardado de `noticias_enriquecido.json`

In [ ]:
def save_enriched(
    data: List[Dict],
    filepath: str,
    errors: Optional[List[Dict]] = None,
) -> None:
    """
    Guarda la lista de noticias enriquecidas en un fichero JSON con
    indentación legible, y opcionalmente un fichero de errores aparte.

    Parameters
    ----------
    data     : List[Dict]  Lista de noticias enriquecidas.
    filepath : str         Ruta del fichero JSON de salida.
    errors   : List[Dict] (opcional) Lista de registros con error.
    """
    with open(filepath, "w", encoding="utf-8") as fh:
        json.dump(data, fh, ensure_ascii=False, indent=2, default=str)
    file_size_mb = Path(filepath).stat().st_size / 1_048_576
    logger.info("Fichero '%s' guardado (%.2f MB).", filepath, file_size_mb)
    print(f"💾 '{filepath}' guardado — {file_size_mb:.2f} MB — {len(data)} registros.")

    if errors:
        err_path = filepath.replace(".json", "_errores.json")
        with open(err_path, "w", encoding="utf-8") as fh:
            json.dump(errors, fh, ensure_ascii=False, indent=2)
        print(f"⚠️  Fichero de errores: '{err_path}' — {len(errors)} registros.")


# ── Guardar resultados ──────────────────────────────────────────────────────
save_enriched(noticias_enriquecidas, OUTPUT_FILE, errors=errores)

# ── Descarga directa en Colab (si se desea) ─────────────────────────────────
# Descomenta las siguientes líneas para descargar el fichero directamente:
# from google.colab import files
# files.download(OUTPUT_FILE)

## 9. 📊 Estadísticas finales

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ESTADÍSTICAS GENERALES
# ─────────────────────────────────────────────────────────────────────────────

total     = len(noticias_enriquecidas)
n_ok      = sum(1 for n in noticias_enriquecidas if n["enrich_status"] == "ok")
n_err     = total - n_ok
pct_ok    = n_ok / total * 100 if total else 0

print("=" * 60)
print("  📊  ESTADÍSTICAS FINALES DEL ENRIQUECIMIENTO")
print("=" * 60)
print(f"  Total de noticias          : {total:>6}")
print(f"  Enriquecidas correctamente : {n_ok:>6}  ({pct_ok:.1f}%)")
print(f"  Con error / sin precio     : {n_err:>6}  ({100 - pct_ok:.1f}%)")
print()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DISTRIBUCIÓN BULLISH / BEARISH / NEUTRAL
# ─────────────────────────────────────────────────────────────────────────────

def reaction_distribution(noticias: List[Dict], field: str, label: str) -> None:
    """
    Imprime la distribución de reacciones de mercado para el campo dado.

    Parameters
    ----------
    noticias : List[Dict]  Lista de noticias enriquecidas.
    field    : str         Nombre del campo de reacción ('market_reaction_1d', etc.).
    label    : str         Etiqueta descriptiva para el horizonte.
    """
    values = [n[field] for n in noticias if n.get(field) is not None]
    counts = pd.Series(values).value_counts()
    total_valid = len(values)
    print(f"  {label} (n={total_valid}):")
    for reaction in ["bullish", "neutral", "bearish"]:
        cnt = counts.get(reaction, 0)
        pct = cnt / total_valid * 100 if total_valid else 0
        bar = "█" * int(pct / 2)
        print(f"    {reaction:<8} : {cnt:>5}  ({pct:>5.1f}%)  {bar}")
    print()


print("─" * 60)
print("  📈  DISTRIBUCIÓN BULLISH / BEARISH / NEUTRAL")
print("─" * 60)
reaction_distribution(noticias_enriquecidas, "market_reaction_1d",  "A  1 día  bursátil ")
reaction_distribution(noticias_enriquecidas, "market_reaction_5d",  "A  5 días bursátiles")
reaction_distribution(noticias_enriquecidas, "market_reaction_20d", "A 20 días bursátiles")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DISTRIBUCIÓN POR ÍNDICE / SECTOR
# ─────────────────────────────────────────────────────────────────────────────

df_stats = pd.DataFrame(noticias_enriquecidas)

print("─" * 60)
print("  🗂️   DISTRIBUCIÓN POR ÍNDICE / SECTOR")
print("─" * 60)
dist_is = df_stats["indice_sector"].value_counts(dropna=False)
for val, cnt in dist_is.items():
    pct = cnt / total * 100
    ticker = get_ticker(val) or "???"
    print(f"  {str(val):<35} ({ticker:<10}): {cnt:>5}  ({pct:.1f}%)")
print()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RETORNOS MEDIOS POR ÍNDICE / SECTOR
# ─────────────────────────────────────────────────────────────────────────────

print("─" * 60)
print("  📉  RETORNOS MEDIOS (%) POR ÍNDICE / SECTOR")
print("─" * 60)

ret_cols = ["return_1d", "return_5d", "return_20d"]
for col in ret_cols:
    df_stats[col] = pd.to_numeric(df_stats[col], errors="coerce")

ret_by_group = (
    df_stats.groupby("indice_sector")[ret_cols]
    .mean()
    .round(4)
)
print(ret_by_group.to_string())
print()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DISTRIBUCIÓN POR ESTADO DE ENRIQUECIMIENTO
# ─────────────────────────────────────────────────────────────────────────────

print("─" * 60)
print("  ⚙️   ESTADO DE ENRIQUECIMIENTO")
print("─" * 60)
status_counts = df_stats["enrich_status"].value_counts()
for st, cnt in status_counts.items():
    pct = cnt / total * 100
    print(f"  {st:<15}: {cnt:>5}  ({pct:.1f}%)")
print()

# ─────────────────────────────────────────────────────────────────────────────
# TOP ERRORES MÁS FRECUENTES (si los hay)
# ─────────────────────────────────────────────────────────────────────────────
if errores:
    print("─" * 60)
    print("  ❌  MUESTRA DE ERRORES (primeros 10)")
    print("─" * 60)
    df_err = pd.DataFrame(errores)
    print(df_err[["idx", "fecha", "indice_sector", "status", "error"]].head(10).to_string(index=False))
    print()

print("=" * 60)
print("  ✅  Pipeline completado correctamente.")
print(f"  Fichero de salida : {OUTPUT_FILE}")
print(f"  Log de errores    : enriquecimiento_errores.log")
print("=" * 60)

## 10. 📥 Descarga de resultados

Descarga a tu ordenador el JSON de noticias ya enriquecidas con los datos de Yahoo
Finance (`OUTPUT_FILE`), el fichero de errores si se generó alguno, y el log
completo del proceso (`enriquecimiento_errores.log`).


In [ ]:
if IN_COLAB:
    print(f"Descargando {OUTPUT_FILE} ...")
    files.download(OUTPUT_FILE)

    _err_path = OUTPUT_FILE.replace(".json", "_errores.json")
    if os.path.exists(_err_path):
        print(f"Descargando {_err_path} ...")
        files.download(_err_path)

    _log_path = "enriquecimiento_errores.log"
    if os.path.exists(_log_path):
        print(f"Descargando {_log_path} ...")
        files.download(_log_path)
else:
    print("Fuera de Colab: los archivos ya están guardados en el directorio actual; "
          f"descárgalos directamente desde ahí ({OUTPUT_FILE}).")
